# Dask Delayed

These functions do simple operations like add two numbers together, but they sleep for a random amount of time to simulate real work.

In [ ]:
from dask.distributed import Client, progress

c = Client()
c

In [ ]:
import time
import random
import dask


@dask.delayed
def inc(x):
    time.sleep(random.random())
    return x + 1

@dask.delayed
def dec(x):
    time.sleep(random.random())
    return x - 1

def add(x, y):
    time.sleep(random.random())
    return x + y

In [ ]:
%%time
x = inc(1)
y = dec(2)
z = add(x, y)
z

In [ ]:
%%time
z.compute()

In [ ]:
z.visualize()

In [ ]:
def inc(x):
    time.sleep(1)
    return x + 1

def dec(x):
    time.sleep(1)
    return x - 1

def add(x, y):
    time.sleep(1)
    return x + y

data = [1, 2, 3, 4, 5, 6, 7, 8]

In [ ]:
%%time

results = []

for x in data:
    y = inc(x)
    results.append(y)
    
total = sum(results)

In [ ]:
total

# Dask natively scales Python

Dask provides advanced parallelism for analytics, enabling performance at scale for the tools you love

In [ ]:
import os
import time


import dask
import dask.array as da
import dask.dataframe as dd
import numpy as np

In [ ]:
print(dask.__version__)
print(np.__version__)

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Reading Dask DataFrame

Read CSV files into a Dask.DataFrame: This parallelizes the `pandas.read_csv()`:
- `blocksize=25e7`: 250MB chunks
- `dtype`: Dask don't explore all the data, sometime it mismatch the dtypes

In [ ]:
dtype = {
    'fine_grained_location': 'float64', 
    'officer_id':'object', 
    'county_fips': 'float64',
    'search_type': 'object',
    'search_type_raw': 'object'
}

ddf = dd.read_csv(
    "/kaggle/input/stanford-open-policing-project-texas/TX_2010_onwards.csv", 
#     blocksize="25MB", 
    dtype=dtype, 
    low_memory=False,
    assume_missing=True
)

# ddf = ddf.map_partitions(cudf.from_pandas)  # convert pandas partitions into cudf partitions

ddf

In [ ]:
%%time
ddf.head()

# What is this Dask Dataframe?
A large, virtual dataframe divided along the index into multiple Pandas dataframes:

In [ ]:
ddf.map_partitions(len).compute()

In [ ]:
%%time
ddf.map_partitions(type).compute()

# Dask DataFrame Shape

In [ ]:
%%time
len(ddf)

In [ ]:
%%time
ddf.shape[0].compute()

In [ ]:
ddf_rows = ddf.shape[0].compute()
ddf_col = ddf.shape[1]
print(f"Number of rows: {ddf_rows}")
print(f"Number of columns: {ddf_col}")

# Missing Values

In [ ]:
%%time
(ddf.isna().sum().compute() / ddf_rows) * 100

# Drop Columns

In [ ]:
columns = [
    'police_department', 'driver_age_raw', 'driver_age', 'search_type_raw', 
    'search_type', 'is_arrested'
]
print(f"Number of columns before removing columns: {ddf.shape[1]}")

ddf = ddf.drop(columns, axis=1)

print(f"Number of columns After removing columns: {ddf.shape[1]}")

In [ ]:
(ddf.isna().sum().compute() / ddf_rows) * 100

# Drop Rows

In [ ]:
print(ddf_rows)
ddf = ddf.dropna(how='any')
print(ddf.shape[0].compute())

In [ ]:
(ddf.isna().sum().compute() / ddf_rows) * 100

# Drop_duplicates Rows

In [ ]:
# print(ddf.shape[0].compute())

# print(ddf.drop_duplicates().shape[0].compute())

In [ ]:
# print(ddf.shape[0].compute())
# # ddf = ddf.drop_duplicates(subset=['id'])
# print(ddf.drop_duplicates(subset=['id']).shape[0].compute())

In [ ]:
ddf['id'].nunique().compute()

# Dask DataFrame info

In [ ]:
non_object_col = ddf.columns[(ddf.dtypes != object) & (ddf.dtypes != bool)].to_list()

In [ ]:
ddf[non_object_col]

In [ ]:
ddf.info()

In [ ]:
# Describe shows only numerical columns
ddf[non_object_col].describe(percentiles=[.25, .5, .75, .85, .9]).compute()

# Analysing some columns

In [ ]:
# for column in ddf.columns:
#     print(f"{column}: Number of unique values {ddf[column].nunique().compute()}")
#     print("_______________________________________________\n")

In [ ]:
ddf.columns

In [ ]:
ddf['contraband_found'].value_counts().compute()

In [ ]:
ddf['stop_outcome'].value_counts().compute()

In [ ]:
ddf['driver_race'].value_counts().compute()

In [ ]:
ddf['driver_gender'].value_counts().compute()

In [ ]:
ddf['state'].value_counts().compute()

In [ ]:
# State contains only one value
ddf = ddf.drop('state', axis=1)

In [ ]:
ddf['county_name'].value_counts().compute()

In [ ]:
montgomery = ddf[ddf['county_name'].str.contains('Montgomery')]
montgomery_county_fips = montgomery.groupby('driver_race').driver_gender.agg(['count'])

In [ ]:
montgomery.groupby('driver_race').driver_gender.count().compute()

In [ ]:
# montgomery_county_fips.visualize()

In [ ]:
# montgomery_county_fips.compute()

# Dask Arrays

In [ ]:
import numpy as np
import dask.array as da

In [ ]:
a_np = np.arange(1, 50, 3)
a_np

In [ ]:
a_da = da.arange(1, 50, 3, chunks=5)
a_da

In [ ]:
print(a_da.dtype)
print(a_da.shape)

In [ ]:
a_da.visualize()

In [ ]:
(a_da ** 2).visualize()

In [ ]:
print(a_da.chunks)
print(a_da.chunksize)

In [ ]:
x = da.random.random(20, chunks=5)
x

In [ ]:
result = x.sum()
result

In [ ]:
result.visualize()

In [ ]:
result.compute()

In [ ]:
x = da.random.random(size=(15, 15), chunks=(10, 5))
x

In [ ]:
print(x.chunks)
print(x.chunksize)

In [ ]:
result = (x + x.T).sum()
result

In [ ]:
result.visualize()

In [ ]:
result.compute()

In [ ]:
x = da.random.random(size=(20_000, 20_000), chunks=(2_000, 2_000))
x

In [ ]:
result = (x + x.T).sum()
result

In [ ]:
result.compute()